In [1]:
# Librerías básicas
import re
import os
import glob
import unicodedata
import docx2txt


In [2]:
# STEP 1: Load documents
# Directory containing the .docx files
DATA_DIR = "../data"

In [3]:
!pip install python-docx



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Install docx2txt if not already installed
!pip install docx2txt

# Find all .docx files in the data directory
docx_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.docx")))
print(f"Found {len(docx_files)} .docx files:")
for f in docx_files:
    print(f"  {os.path.basename(f)}")

# Read and concatenate all documents
all_texts = []
for filepath in docx_files:
    doc_text = docx2txt.process(filepath)
    all_texts.append(doc_text)
    print(f"Read: {os.path.basename(filepath)} ({len(doc_text)} chars)")

# Combine all texts into a single string
text = "\n\n".join(all_texts)

print(f"\nTotal length: {len(text)} chars across {len(docx_files)} files")


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Found 19 .docx files:
  243591-DO-AVO-11-V07-251210.docx
  252562-DO-AVO-02-V01.docx
  252562-DO-AVO-03-V01.docx
  252562-DO-AVO-04-V01.docx
  252562-DO-AVO-05-V01.docx
  252562-DO-AVO-06-V01.docx
  252562-DO-AVO-07-V01.docx
  254275-DO-AVO-14-V07.docx
  254275-DO-AVO-15-V07.docx
  254275-DO-AVO-16-V07.docx
  254275-DO-AVO-17-V07.docx
  254275-DO-AVO-18-V07.docx
  254275-DO-AVO-19-V07.docx
  254275-DO-AVO-20-V07.docx
  254275-DO-AVO-21-V01.docx
  254275-DO-AVO-22-V01.docx
  254275-DO-AVO-23-V01.docx
  254275-DO-AVO-24-V01.docx
  254275-DO-AVO-28-V01.docx
Read: 243591-DO-AVO-11-V07-251210.docx (25644 chars)
Read: 252562-DO-AVO-02-V01.docx (8979 chars)
Read: 252562-DO-AVO-03-V01.docx (14426 chars)
Read: 252562-DO-AVO-04-V01.docx (11635 chars)
Read: 252562-DO-AVO-05-V01.docx (8593 chars)
Read: 252562-DO-AVO-06-V01.docx (10629 chars)
Read: 252562-DO-AVO-07-V01.docx (8008 chars)
Read: 254275-DO-AVO-14-V07.docx (11558 chars)
Read: 254275-DO-AVO-15-V07.docx (12401 chars)
Read: 254275-DO-AVO-1

In [5]:
# STEP 2: Clean text
def clean_text(text):
    # 1. Unicode normalization: resolves ligatures, full-width chars, composite characters
    text = unicodedata.normalize("NFKC", text)

    # 2. Remove control characters (null bytes, form feeds, etc.) but keep \n and \t
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)

    # 3. Normalize dash variants (em dash, en dash, soft hyphen, minus) to hyphen
    text = re.sub(r'[­‐-―−－]', '-', text)

    # 4. Normalize typographic quotes to straight ASCII quotes
    text = re.sub(r'[‘’‚‛]', "'", text)
    text = re.sub(r'[“”„‟]', '"', text)

    # 5. Replace bullet-point characters with a dash
    text = re.sub(r'[•‣●◦⁃∙·]', '-', text)

    # 6. Remove standalone page numbers (lines containing only digits)
    text = re.sub(r'(?m)^\s*\d{1,4}\s*$', '', text)

    # 7. Remove repeated punctuation runs (e.g. "---", "___", "...")
    text = re.sub(r'([.\-_=])\1{2,}', r'\1', text)

    # 8. Fix spacing before punctuation (e.g. "word ." → "word.")
    text = re.sub(r'\s+([.,;:!?)])', r'\1', text)

    # 9. Collapse all whitespace (spaces, tabs, newlines) to a single space
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [6]:
cleaned_document_text = clean_text(text)
print("Longitud del texto limpio:", len(cleaned_document_text))
print(cleaned_document_text[0:500])  # Print the first 500 characters of the cleaned text

Longitud del texto limpio: 266869
ACTA DE VISITA DE OBRA No 010 243591-DO-AVO-011-V07_251210 MEJORA DE LA ACCESIBILIDAD DE LA ESTACIÓN DE SANTA PERPÈTUA DE LA MOGODA - LA FLORIDA, BARCELONA Fecha Lugar Hora de inicio Hora de Finalización 10/12/25 Estación de Santa Perpètua de la Mogoda - La Florida. 10:00 13:00 Principales asistentes: Nombre Empresa Cargo E-mail Teléfono Firma opcional FUNCIÓN: DC (director de contrato) DF (dirección Facultativa) DO (dirección de obra) DEO (dirección de ejecución) UTE (contratista) CSS (coordina


In [7]:
# STEP 3: Chunking
def chunk_text(text, chunk_size=500, overlap=100, min_chunk_size=50):
    """
    Split text into overlapping chunks, snapping boundaries to sentences then words.

    Args:
        chunk_size:     target maximum characters per chunk
        overlap:        characters of context shared between consecutive chunks
        min_chunk_size: chunks shorter than this are discarded (avoids tiny tail chunks)
    """
    chunks = []
    if not text:
        return chunks

    stride = chunk_size - overlap
    start = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + chunk_size, text_len)

        if end < text_len:
            # 1. Try to end at a sentence boundary in the last 20 % of the window
            search_from = start + int(chunk_size * 0.8)
            best = -1
            for punct in '.!?;':
                pos = text.rfind(punct, search_from, end)
                if pos > best:
                    best = pos
            if best != -1:
                end = best + 1          # include the closing punctuation
            else:
                # 2. Fall back to the nearest word boundary
                space = text.rfind(' ', search_from, end)
                if space != -1:
                    end = space         # cut just before the space

        chunk = text[start:end].strip()
        if len(chunk) >= min_chunk_size:
            chunks.append(chunk)

        start += stride
        # Snap start forward so the next chunk never begins mid-word
        if start < text_len and start > 0 and text[start - 1] not in ' \n':
            nxt = text.find(' ', start)
            start = nxt + 1 if nxt != -1 else text_len

    return chunks

In [8]:
chunks = chunk_text(cleaned_document_text)
print(f"Número total de chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)

Número total de chunks: 662

--- Chunk 1 ---
ACTA DE VISITA DE OBRA No 010 243591-DO-AVO-011-V07_251210 MEJORA DE LA ACCESIBILIDAD DE LA ESTACIÓN DE SANTA PERPÈTUA DE LA MOGODA - LA FLORIDA, BARCELONA Fecha Lugar Hora de inicio Hora de Finalización 10/12/25 Estación de Santa Perpètua de la Mogoda - La Florida. 10:00 13:00 Principales asistentes: Nombre Empresa Cargo E-mail Teléfono Firma opcional FUNCIÓN: DC (director de contrato) DF (dirección Facultativa) DO (dirección de obra) DEO (dirección de ejecución) UTE (contratista) CSS

--- Chunk 2 ---
Facultativa) DO (dirección de obra) DEO (dirección de ejecución) UTE (contratista) CSS (coordinador de seguridad) OBJETO Visita Semanal de obra DOCUMENTACIÓN ENTREGADA O A ENTREGAR DOCUMENTACIÓN DE CONSULTA Proyecto Constructivo de Mejora de la accesibilidad de la estación de Santa Perpetua de la Mogoda - La Florida, Barcelona. ASUNTOS TRATADOS, ACUERDOS ALCANZADOS, COMUNICACIONES, INSTRUCCIONES: No Fecha DESCRIPCIÓN DEL ASUNTO: 1. AVANCE DE L

In [9]:
# Dataset estructurado
data = []

for i, chunk in enumerate(chunks):
    data.append({
        "chunk_id": i,
        "text": chunk
    })



In [10]:
print(len(data))
# 518 chunks

662


In [11]:
# Para ver el Data
for item in data:
    print(item)
    print("\n---\n")

{'chunk_id': 0, 'text': 'ACTA DE VISITA DE OBRA No 010 243591-DO-AVO-011-V07_251210 MEJORA DE LA ACCESIBILIDAD DE LA ESTACIÓN DE SANTA PERPÈTUA DE LA MOGODA - LA FLORIDA, BARCELONA Fecha Lugar Hora de inicio Hora de Finalización 10/12/25 Estación de Santa Perpètua de la Mogoda - La Florida. 10:00 13:00 Principales asistentes: Nombre Empresa Cargo E-mail Teléfono Firma opcional FUNCIÓN: DC (director de contrato) DF (dirección Facultativa) DO (dirección de obra) DEO (dirección de ejecución) UTE (contratista) CSS'}

---

{'chunk_id': 1, 'text': 'Facultativa) DO (dirección de obra) DEO (dirección de ejecución) UTE (contratista) CSS (coordinador de seguridad) OBJETO Visita Semanal de obra DOCUMENTACIÓN ENTREGADA O A ENTREGAR DOCUMENTACIÓN DE CONSULTA Proyecto Constructivo de Mejora de la accesibilidad de la estación de Santa Perpetua de la Mogoda - La Florida, Barcelona. ASUNTOS TRATADOS, ACUERDOS ALCANZADOS, COMUNICACIONES, INSTRUCCIONES: No Fecha DESCRIPCIÓN DEL ASUNTO: 1. AVANCE DE LA OB

In [12]:
#  Guarda el resultado
import json

with open("chunks_acta.json", "w") as f:
    json.dump(data, f, indent=4)

Segundo paso. Crear embeddings con BERT

In [13]:
# Importar BERT
!pip install transformers torch

from transformers import AutoTokenizer, AutoModel
import torch



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\REPOS\ML problems\POSTGRAU\RAG-UPCSchool-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
# Descargar el modelo
model_name = "BSC-LT/MrBERT-es"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 6486.59it/s]
[transformers] ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
# Aplicar el modelo a los chunks
embeddings = []

for item in data:
    inputs = tokenizer(item["text"], return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    embedding = outputs.last_hidden_state[:, 0, :].squeeze().tolist()

    embeddings.append({
        "chunk_id": item["chunk_id"],
        "embedding": embedding,
        "text": item["text"]
    })

In [17]:
# Define your search query here
search_query = "instalaciones de sistemas y megafonía" # @param {type:"string"}

In [18]:
!pip install faiss-cpu



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import faiss
import numpy as np

# Build FAISS index from embeddings
embedding_vectors = [item['embedding'] for item in embeddings]
dimension = len(embedding_vectors[0])
embedding_array = np.array(embedding_vectors).astype('float32')
index = faiss.IndexFlatL2(dimension)
index.add(embedding_array)
print(f"FAISS index created with {index.ntotal} embeddings.")

# Embed the search query
query_inputs = tokenizer(search_query, return_tensors="pt", truncation=True, padding=True)

with torch.no_grad():
    query_outputs = model(**query_inputs)

query_embedding = query_outputs.last_hidden_state[:, 0, :].squeeze().numpy().astype('float32')
query_embedding = query_embedding.reshape(1, -1)

# Perform similarity search
k = 5
distances, indices = index.search(query_embedding, k)

print(f"Top {k} most similar chunks for query: '{search_query}'")
for i, idx in enumerate(indices[0]):
    print(f"\n--- Rank {i+1} (Distance: {distances[0][i]:.4f}) ---")
    print(f"Chunk ID: {embeddings[idx]['chunk_id']}")
    print(f"Text: {embeddings[idx]['text']}")

FAISS index created with 662 embeddings.
Top 5 most similar chunks for query: 'instalaciones de sistemas y megafonía'

--- Rank 1 (Distance: 1059.4391) ---
Chunk ID: 479
Text: AR-33 Ascensores. Requisitos técnicos mínimos La DEO solicita a la constructora que en relación a los mínimos requisitos técnicos que tienen que cumplir los ascensores, la constructora verifique que, su propuesta de ascensores cumple estos requisitos técnicos mínimos, y justifique a la Dirección Facultativa del cumplimiento de cada uno de ellos. AR-34 Muro andén drenaje trasdós DF recuerda a la constructora que envié evidencias fotográficas del tubo de drenaje instalado.

--- Rank 2 (Distance: 1063.9783) ---
Chunk ID: 558
Text: estructura metálica Durante la visita de obra, y durante la inspección de la estructura metálica instalada esta semana, se observa que el pilar P-08 tiene un perno tocando la pletina y otro perno contiguo en posición muy próxima a la pletina. Esto ha provocado que la se ha recortado ligera

The embeddings have been successfully stored in a FAISS index. You can now perform similarity searches on this index.

## Executive Summary of Document Analysis

This analysis involved extracting, processing, and indexing text from a provided DOCX document for efficient information retrieval. The key steps and findings are as follows:

### 1. Document Extraction and Cleaning
- The document **"243591-DO-AVO-02-V01-251008.docx"** was successfully loaded and its text content extracted.
- A cleaning process was applied to remove excess whitespace, resulting in a refined text of **7100 characters** (reduced from 7434 characters).

### 2. Text Chunking
- The cleaned text was divided into **16 overlapping chunks** (each with a size of 500 characters and an overlap of 50 characters) to facilitate granular analysis.
- These chunks were structured with `doc_id` and `chunk_id` for easy identification.

### 3. Embeddings Generation
- The **`BSC-LT/MrBERT-es`** transformer model was used to generate dense vector embeddings for each text chunk. These embeddings capture the semantic meaning of each chunk.

### 4. Vector Database (FAISS) Setup
- A **FAISS `IndexFlatL2` index** was created and populated with the 16 generated embeddings. This enables rapid similarity searches against the document's content.

### 5. Similarity Search Results
- A query, **"instalaciones de sistemas y megafonía"**, was embedded and used to search the FAISS index for the most relevant text segments. The top 5 results were:
    - **Rank 1 (Chunk ID 6):** Mentions **`Desmontaje acopio y o retirada de instalaciones anti-intrusión, megafonía y cartelería de señalización informativa de andén.`**
    - **Rank 2 (Chunk ID 3):** Discusses **`Implantación de obra: Montaje de casetas e instalaciones de obra y su señalización.`**
    - **Rank 3 (Chunk ID 11):** Refers to **`Conexión de acometida pluvial`**, which is less directly related but might share contextual terms.
    - **Rank 4 (Chunk ID 7):** Details **`Desmontaje acopio y o retirada de instalaciones anti-intrusión, megafonía y cartelería de señalización informativa de andén.`**
    - **Rank 5 (Chunk ID 2):** Mentions **`Implantación de obra: Montaje de casetas e instalaciones de obra`**, similar to Rank 2.

These results indicate that the system successfully identified the most semantically relevant sections of the document related to the query, particularly those explicitly mentioning 'megafonía' and 'instalaciones'.

In [20]:
display(embeddings[:2])

[{'chunk_id': 0,
  'embedding': [-0.14772294461727142,
   1.0658923387527466,
   0.4840186536312103,
   0.1801847517490387,
   -0.15692563354969025,
   -0.21545341610908508,
   -1.014114499092102,
   1.6321057081222534,
   0.07111398875713348,
   -0.1936967819929123,
   0.026681402698159218,
   0.3605414927005768,
   0.7870900630950928,
   -0.255474716424942,
   -0.5118705630302429,
   -0.14679548144340515,
   1.719712734222412,
   0.04149860143661499,
   0.5515087246894836,
   -0.9297316074371338,
   0.4081115126609802,
   0.885692834854126,
   0.10095789283514023,
   -0.19164489209651947,
   -0.49260199069976807,
   0.9942902326583862,
   -0.26012086868286133,
   -0.833245575428009,
   -0.8701753616333008,
   -1.5522491931915283,
   0.47967812418937683,
   -0.8221530914306641,
   1.0035020112991333,
   -0.1543070673942566,
   -44.362361907958984,
   -0.5847112536430359,
   -1.218592643737793,
   1.4519093036651611,
   -0.5935711860656738,
   1.0694540739059448,
   -1.4196339845657349

In [21]:
import json

with open('chunks_acta.json', 'r') as f:
    loaded_data = json.load(f)

# You can now use 'loaded_data' for further processing.
# For example, to verify its content:
print(f"Loaded {len(loaded_data)} chunks from 'chunks_acta.json'")
# display(loaded_data[:2]) # Display first two items to check

Loaded 662 chunks from 'chunks_acta.json'


In [ ]:
!pip install chromadb


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import chromadb

# Persistent client — saves to disk in notebooks/chroma_db/
client = chromadb.PersistentClient(path="./chroma_db")

# Delete collection if it already exists (to avoid duplicate insertions on re-runs)
if "actas_obra" in [c.name for c in client.list_collections()]:
    client.delete_collection("actas_obra")

# No built-in embedding function — we provide our own MrBERT embeddings
collection = client.create_collection(
    name="actas_obra",
    metadata={"hnsw:space": "l2"}
)

collection.add(
    ids=[str(item["chunk_id"]) for item in embeddings],
    embeddings=[item["embedding"] for item in embeddings],
    documents=[item["text"] for item in embeddings],
    metadatas=[{"chunk_id": item["chunk_id"]} for item in embeddings]
)

print(f"Stored {collection.count()} chunks in ChromaDB.")

Stored 662 chunks in ChromaDB.


## Paso 4: Almacenar embeddings en ChromaDB (persistente)